# lesson10 homework｜Selenium Notebook

先從上到下執行設定和函式定義格。每次示範或檢查會啟動練習站及 Chrome，結束時會關閉；請等待執行完成後再執行下一格。


In [ ]:

import os
import sys
import time
from collections import Counter
from pathlib import Path

import os
from pathlib import Path

# 從教材根目錄或本堂資料夾啟動 Notebook 都可以。
current = Path.cwd().resolve()
lesson_name = 'lesson10_scroll_and_load_more'
candidates = [current, *current.parents]
lesson_path = next((p for p in candidates if p.name == lesson_name and (p / 'fixtures').is_dir()), None)
if lesson_path is None:
    lesson_path = next((p / 'lessons' / lesson_name for p in candidates if (p / 'lessons' / lesson_name / 'fixtures').is_dir()), None)
if lesson_path is None:
    raise FileNotFoundError('找不到本堂教材。請用 VS Code 開啟完整教材資料夾，重新啟動核心後再執行。')
os.chdir(lesson_path)
print('工作資料夾：', Path.cwd())
sys.path.insert(0, str(Path.cwd().parents[1] / "tools"))

import pandas as pd
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

from practice_site import POSTS, start_server, stop_server

HEADLESS = os.environ.get("HEADLESS") == "1"




## new_driver


In [ ]:
def new_driver():
    options = webdriver.ChromeOptions()
    if HEADLESS:
        options.add_argument("--headless=new")
    options.add_argument("--window-size=1280,900")
    return webdriver.Chrome(options=options)


## post_count


In [ ]:
def post_count(driver):
    """已經寫好的小工具：目前頁面上有幾篇文章。"""
    return len(driver.find_elements(By.CSS_SELECTOR, "article.post"))


## scroll_to_bottom


In [ ]:
def scroll_to_bottom(driver):
    """已經寫好的小工具：把頁面捲到最底。"""
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")


## h1_first_titles


In [ ]:
def h1_first_titles(driver, base_url, n):
    """打開 base_url + "/feed"，捲動到『文章數達到 n 篇以上』就停（不要全部載入），
    回傳前 n 篇文章的標題（h3.post-title 的文字），list。迴圈最多跑 30 次。
    提示：網頁每次載入 10 篇，所以停下來時頁面上的文章數可能比 n 多。"""
    # TODO: 請在這裡寫程式
    return None


## h2_load_more_tag_counts


In [ ]:
def h2_load_more_tag_counts(driver, base_url):
    """打開 base_url + "/load-more"，一直按「載入更多」直到按鈕消失，
    統計每個標籤（span.post-tag）有幾篇，回傳 dict，例如 {"爬蟲": 3, ...}。"""
    # TODO: 請在這裡寫程式
    return None


## h3_find_api


In [ ]:
def h3_find_api(driver, base_url):
    """打開 /feed，等第一批文章出現，用 JavaScript 問瀏覽器網頁呼叫過哪些網址：
        driver.execute_script("return performance.getEntriesByType('resource').map(e => e.name)")
    找出第一個包含 "/api/" 的網址，去掉開頭的 base_url，存成 api_path（例如 "/api/feed?offset=0&limit=10"）。
    再用 requests 呼叫同一個 API，但把 limit 改成 100，一次拿全部，數一數共有幾篇。
    回傳 (api_path, 文章數) 這個 tuple。"""
    # TODO: 請在這裡寫程式
    return None


## check_h1


In [ ]:
def check_h1(driver, base_url):
    titles = h1_first_titles(driver, base_url, 25)
    expected = [post["title"] for post in POSTS[:25]]
    assert titles == expected, "前 25 篇標題不對"
    assert post_count(driver) == 30, f"應該在載入 30 篇時就停（第 3 批），頁面上實際有 {post_count(driver)} 篇"


## check_h2


In [ ]:
def check_h2(driver, base_url):
    counts = h2_load_more_tag_counts(driver, base_url)
    expected = dict(Counter(post["tag"] for post in POSTS[:25]))
    assert counts == expected, f"{counts} 應該是 {expected}"


## check_h3


In [ ]:
def check_h3(driver, base_url):
    result = h3_find_api(driver, base_url)
    assert result == ("/api/feed?offset=0&limit=10", 60), result


## main


In [ ]:
def main():
    checks = [
        ("H1", "提早停止，取前 25 篇", check_h1),
        ("H2", "按到底並統計標籤", check_h2),
        ("H3", "找出 API 並一次抓完", check_h3),
    ]

    server, base_url = start_server()
    driver = None
    failed = 0
    try:
        driver = new_driver()
        for number, description, check in checks:
            driver.get("about:blank")
            driver.delete_all_cookies()
            started = time.time()
            try:
                check(driver, base_url)
                passed, note = True, ""
            except Exception as error:  # noqa: BLE001 - show every kind of mistake
                passed = False
                note = f"{type(error).__name__}: {str(error).splitlines()[0] if str(error) else ''}"
            failed += 0 if passed else 1
            print(f"{number} {'PASS' if passed else 'FAIL'}  {description}  ({time.time() - started:.1f}s) {note}")
    finally:
        if driver is not None:
            driver.quit()
        stop_server(server)

    print(f"\n通過 {len(checks) - failed} / {len(checks)} 題")
    sys.exit(1 if failed else 0)


## 執行檢查
完成上方 TODO 並執行相關函式格後，再執行本格。


In [ ]:
try:
    main()
except SystemExit as result:
    print("檢查結束；狀態：", result.code)
